In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import os
import yaml

plt.close('all')

## mAP per Class: YOLOv5s Baseline vs Pruned

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
test_folder = '../runs/test/'
data_yaml   = '../data/coco.yaml'

# ── Load COCO class names ──────────────────────────────────────────────────
with open(data_yaml) as f:
    coco_data = yaml.safe_load(f)
class_names = coco_data['names']   # list of 80 strings
nc = len(class_names)

# ── Helper: parse results.txt ──────────────────────────────────────────────
def parse_results(path):
    """Return (overall_map, per_class_map_array) from results.txt."""
    values = []
    with open(path, 'r') as f:
        for line in f:
            for token in line.split():
                clean = token.replace('[', '').replace(']', '')
                try:
                    values.append(float(clean))
                except ValueError:
                    pass
    overall = values[0]
    per_cls = np.array(values[1:1 + nc])
    return overall, per_cls

# ── Read baseline and pruned ───────────────────────────────────────────────
base_overall, base_per     = parse_results(os.path.join(test_folder, 'bench_baseline', 'results.txt'))
pruned_overall, pruned_per = parse_results(os.path.join(test_folder, 'bench_pruned',   'results.txt'))

# ── Derived arrays (defined here so all cells below can use them) ──────────
x         = np.arange(nc)                                              # class indices
drop      = base_per - pruned_per                                      # absolute mAP drop
retention = np.where(base_per > 0, pruned_per / base_per * 100, 100.0) # retention %

print(f"Baseline  mAP@0.5:0.95 = {base_overall:.4f}")
print(f"Pruned    mAP@0.5:0.95 = {pruned_overall:.4f}")
print(f"Drop                   = {base_overall - pruned_overall:.4f}  "
      f"({(base_overall - pruned_overall)/base_overall*100:.2f}%)")
print(f"Retention              = {pruned_overall/base_overall*100:.2f}%")

## Per-Class mAP Bar Chart

In [ ]:
# ── Bar chart: per-class mAP@0.5:0.95 ─────────────────────────────────────
width = 0.4

fig, ax = plt.subplots(figsize=(20, 5))
ax.bar(x - width/2, base_per,   width, label=f'Baseline  ({base_overall:.4f})',   color='steelblue',  alpha=0.85)
ax.bar(x + width/2, pruned_per, width, label=f'Pruned    ({pruned_overall:.4f})', color='darkorange', alpha=0.85)

ax.set_xlabel('Class', fontsize=11)
ax.set_ylabel('mAP@0.5:0.95', fontsize=11)
ax.set_title('Per-Class mAP: YOLOv5s Baseline vs Pruned', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=90, fontsize=7)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))
ax.set_ylim(0, max(base_per.max(), pruned_per.max()) * 1.1)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Per-Class mAP Drop (Baseline − Pruned)

In [ ]:
# ── Bar chart: absolute drop per class ────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 4))
colors = ['red' if d > 0 else 'green' for d in drop]
ax.bar(x, drop, color=colors, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)

ax.set_xlabel('Class', fontsize=11)
ax.set_ylabel('mAP Drop (Baseline − Pruned)', fontsize=11)
ax.set_title('Per-Class mAP Drop after Pruning (YOLOv5s)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=90, fontsize=7)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.3f'))
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"\nTop 5 classes with highest mAP drop:")
top5_idx = np.argsort(drop)[::-1][:5]
for i in top5_idx:
    print(f"  {class_names[i]:<20s}  drop = {drop[i]:.4f}")

## Per-Class mAP Retention (%)

In [ ]:
# ── Retention ratio per class ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 4))
bar_colors = ['green' if r >= 90 else ('orange' if r >= 75 else 'red') for r in retention]
ax.bar(x, retention, color=bar_colors, alpha=0.8)
ax.axhline(100, color='black', linewidth=0.8, linestyle='--')

ax.set_xlabel('Class', fontsize=11)
ax.set_ylabel('mAP Retention (%)', fontsize=11)
ax.set_title('Per-Class mAP Retention after Pruning (YOLOv5s)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=90, fontsize=7)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 110)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"\nOverall mAP retention: {pruned_overall/base_overall*100:.2f}%")
print(f"Classes with retention < 75%: {sum(retention < 75)}")
print(f"Classes with retention >= 90%: {sum(retention >= 90)}")

## Summary Table

In [ ]:
# ── DataFrame summary ─────────────────────────────────────────────────────
df = pd.DataFrame({
    'Class':         class_names,
    'Baseline mAP':  base_per.round(4),
    'Pruned mAP':    pruned_per.round(4),
    'Drop':          drop.round(4),
    'Retention (%)': retention.round(2)
})

df_sorted = df.sort_values('Drop', ascending=False)
print(df_sorted.to_string(index=False))